# Evaluation on Binarized MNIST

Load models and data → precompute FID statistics → sample and evaluate.
The structure follows the [official SFM evaluation notebook](https://github.com/ccr-cheng/statistical-flow-matching/blob/main/eval_bmnist.ipynb).

Run the cells in order from the repository root, using the `ReSFlow` environment.
All checkpoints, sampler settings, seeds, and output paths are configured in
`configs/bmnist_eval_notebook.yml`. The checkpoints must already exist;
this notebook does not train models. Set `evaluation.split` to `valid` for
checkpoint comparisons and to `test` for final evaluation.


## 1. Load models and data


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision.utils import make_grid, save_image
from matplotlib import pyplot as plt
from IPython.display import display
import pandas as pd
from tqdm.auto import tqdm

from configuration import load_config
from datasets import BinaryMNIST
from models import get_flow_model
from evaluation.inception import InceptionV3
from evaluation.fid import calculate_frechet_distance
from utils import seed_all

CONFIG_PATH = Path('configs/bmnist_eval_notebook.yml')
config = load_config(CONFIG_PATH)
eval_cfg = config.evaluation
device = torch.device(eval_cfg.device)
if device.type == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Set evaluation.device: cpu in the YAML for a CPU run.')
if eval_cfg.split not in ('valid', 'test'):
    raise ValueError('evaluation.split must be valid or test')
for key in ('data_batch_size', 'fid_batch_size', 'total_sample', 'sample_batch_size',
            'grid_n_sample', 'grid_nrow'):
    if eval_cfg[key] <= 0:
        raise ValueError(f'evaluation.{key} must be positive')
if eval_cfg.total_sample < 2 or not eval_cfg.seeds or not eval_cfg.samplers or not eval_cfg.checkpoints:
    raise ValueError('FID needs at least two generated images, seeds, samplers, and checkpoints')
if eval_cfg.num_workers < 0 or eval_cfg.warmup_sample < 0:
    raise ValueError('num_workers and warmup_sample must be nonnegative')
for sampler in eval_cfg.samplers:
    if sampler.method not in ('euler', 'ode') or sampler.n_steps < 1:
        raise ValueError(f'Invalid sampler: {sampler}')
for entries in (eval_cfg.checkpoints, eval_cfg.samplers):
    names = [entry.name for entry in entries]
    if len(names) != len(set(names)) or any(Path(name).name != name for name in names):
        raise ValueError('Model and sampler names must be unique filenames')
for entry in eval_cfg.checkpoints:
    if not Path(entry.path).is_file():
        raise FileNotFoundError(f'Checkpoint missing: {entry.path}. Update the YAML before running.')
output_dir = Path(eval_cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Device: {device}; reference split: {eval_cfg.split}; outputs: {output_dir}')


In [ ]:
def load_model(entry):
    checkpoint = torch.load(entry.path, map_location='cpu', weights_only=False)
    # Reflow checkpoints may contain only model weights; fall back to the YAML.
    model_config = checkpoint.get('config', config)
    model = get_flow_model(model_config['model'], model_config['encoder'])
    model.load_state_dict(checkpoint['model'], strict=True)
    model = model.to(device).eval()
    if model.n_class != 2 or model.total_data_dim != 784:
        raise ValueError(f'{entry.name} is not a binarized MNIST model')
    return model

models = {entry.name: load_model(entry) for entry in eval_cfg.checkpoints}
# Missing .amat files are downloaded and converted to .npy automatically.
# Only the split used for this evaluation is prepared.
reference_set = BinaryMNIST(config.datasets.root, split=eval_cfg.split)
reference_loader = DataLoader(
    reference_set, batch_size=eval_cfg.data_batch_size,
    shuffle=False, num_workers=eval_cfg.num_workers,
    pin_memory=device.type == 'cuda',
)
print(f'Loaded {list(models)}; {len(reference_set):,} reference images')


## 2. Precompute FID statistics

Use the same pretrained FID Inception network as SFM. Its weights are downloaded
on first use. Binary images are replicated into three channels. Reference
statistics use every image in the configured split and are cached separately
for `valid` and `test`. Features are collected in batches without loading all
reference images onto the GPU.


In [ ]:
def binary_images(probabilities):
    return (probabilities[..., 0] > probabilities[..., 1]).float().reshape(-1, 1, 28, 28)

@torch.no_grad()
def image_features(images, inception, device, batch_size):
    features = []
    for batch in images.split(batch_size):
        prediction = inception(batch.to(device).expand(-1, 3, -1, -1))[0]
        features.append(prediction.flatten(1).cpu().numpy())
    return np.concatenate(features, axis=0)

def feature_statistics(features):
    if len(features) < 2:
        raise ValueError('FID covariance requires at least two images')
    return np.mean(features, axis=0), np.cov(features, rowvar=False)

@torch.no_grad()
def reference_statistics(loader, inception, device, batch_size, cache_path, metadata):
    signature = json.dumps(metadata, sort_keys=True)
    if cache_path.is_file():
        with np.load(cache_path, allow_pickle=False) as cache:
            if str(cache['metadata'].item()) == signature:
                print(f'Using cached reference statistics: {cache_path}')
                return cache['mu'].copy(), cache['sigma'].copy()
    features = []
    for probabilities, *_ in tqdm(loader, desc='Reference FID features'):
        features.append(image_features(binary_images(probabilities), inception, device, batch_size))
    mu, sigma = feature_statistics(np.concatenate(features, axis=0))
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(cache_path, mu=mu, sigma=sigma, metadata=signature)
    print(f'Saved reference statistics: {cache_path}')
    return mu, sigma


In [ ]:
fid_network = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]]).to(device).eval()
data_path = Path(config.datasets.root) / f'binarized_mnist_{eval_cfg.split}.npy'
data_stat = data_path.stat()
reference_metadata = {
    'split': eval_cfg.split, 'count': len(reference_set),
    'data_path': str(data_path.resolve()), 'size': data_stat.st_size,
    'mtime_ns': data_stat.st_mtime_ns,
    'features': 'pytorch-fid-inception-2048', 'decoder': 'channel0>channel1',
}
mu_ref, sigma_ref = reference_statistics(
    reference_loader, fid_network, device, eval_cfg.fid_batch_size,
    output_dir / f'bmnist_{eval_cfg.split}_fid.npz', reference_metadata,
)


## 3. Sample and evaluate

Evaluate each checkpoint with one-step and adaptive ODE sampling for every
configured seed. Spherical one-step sampling uses the exponential map.
Report FID, sampling time, and the measured number of vector-field evaluations
per batch. The timer excludes warm-up, CPU transfer, FID calculation, and image
export. ODE NFE is measured during integration; `n_steps` does not fix it.
Sample grids and per-seed results are saved to the configured output directory.


In [ ]:
def synchronize(device):
    if device.type == 'cuda':
        torch.cuda.synchronize(device)

@torch.no_grad()
def sample_images(model, sampler, total_sample, batch_size, device, seed, warmup_sample=0):
    model.eval()
    if warmup_sample:
        seed_all(seed)
        model.sample(sampler.method, warmup_sample, sampler.n_steps, device)
        synchronize(device)
    # Reset after warm-up so its draws do not change the evaluated samples.
    seed_all(seed)
    images, nfe_by_batch, elapsed = [], [], 0.0
    calls = [0]
    def count_call(module, inputs):
        calls[0] += 1
    handle = model.register_forward_pre_hook(count_call)
    try:
        for start in tqdm(range(0, total_sample, batch_size), desc=f'Sampling {sampler.name}'):
            take = min(batch_size, total_sample - start)
            calls[0] = 0
            synchronize(device)
            begin = time.perf_counter()
            probabilities = model.sample(sampler.method, take, sampler.n_steps, device)
            synchronize(device)
            elapsed += time.perf_counter() - begin
            nfe_by_batch.append(calls[0])
            images.append(binary_images(probabilities).cpu())
    finally:
        handle.remove()
    return torch.cat(images), elapsed, nfe_by_batch


In [ ]:
rows = []
for entry in eval_cfg.checkpoints:
    model = models[entry.name]
    for sampler in eval_cfg.samplers:
        for seed_index, seed in enumerate(eval_cfg.seeds):
            images, elapsed, nfe = sample_images(
                model, sampler, eval_cfg.total_sample, eval_cfg.sample_batch_size,
                device, seed, eval_cfg.warmup_sample,
            )
            mu_gen, sigma_gen = feature_statistics(
                image_features(images, fid_network, device, eval_cfg.fid_batch_size)
            )
            fid = calculate_frechet_distance(mu_gen, sigma_gen, mu_ref, sigma_ref)
            row = {
                'model': entry.name, 'checkpoint': entry.path, 'sampler': sampler.name,
                'method': sampler.method, 'split': eval_cfg.split, 'seed': int(seed),
                'n_samples': len(images), 'fid': float(fid),
                'sampling_seconds': elapsed, 'seconds_per_1000': elapsed * 1000 / len(images),
                'nfe_mean_per_batch': float(np.mean(nfe)),
                'nfe_min_per_batch': min(nfe), 'nfe_max_per_batch': max(nfe),
            }
            rows.append(row)
            grid = make_grid(images[:eval_cfg.grid_n_sample], nrow=eval_cfg.grid_nrow,
                             padding=2, pad_value=0)
            save_image(grid, output_dir / f'{entry.name}_{sampler.name}_seed{seed}.png')
            print(f"{entry.name} / {sampler.name} / seed={seed}: FID={fid:.4f}; "
                  f"time={elapsed:.3f}s; NFE/batch={np.mean(nfe):.1f}")
            if seed_index == 0:
                plt.figure(figsize=(8, 8))
                plt.imshow(grid.permute(1, 2, 0).numpy())
                plt.title(f'{entry.name} — {sampler.name}')
                plt.axis('off')
                plt.show()
            # Save after each run so earlier results survive a later interruption.
            pd.DataFrame(rows).to_csv(output_dir / 'metrics.csv', index=False)
            (output_dir / 'metrics.json').write_text(json.dumps(rows, indent=2)+'\n')

results = pd.DataFrame(rows)
display(results)
summary = results.groupby(['model', 'sampler', 'split'], sort=False).agg(
    fid_mean=('fid', 'mean'), fid_std=('fid', 'std'),
    seconds_per_1000_mean=('seconds_per_1000', 'mean'),
    seconds_per_1000_std=('seconds_per_1000', 'std'),
    nfe_mean_per_batch=('nfe_mean_per_batch', 'mean'),
).reset_index()
summary.to_csv(output_dir / 'summary.csv', index=False)
(output_dir / 'evaluation_config.json').write_text(json.dumps(config, indent=2)+'\n')
display(summary)
print(f'Saved metrics, summary, configuration, and sample grids to {output_dir}')
